# 🤖 Part 5: Damped Least Squares 수치 역기구학

다자유도 로봇의 **수치적 역기구학(Numerical IK)** 및 특이점(Singularity) 발산 문제를 해결하는 **감쇠 최소자승법(Damped Least Squares, DLS)** 실습이다.

## 1. 수치 역기구학의 기본 루프

목표와 현재 위치 오차 $\mathbf{e}$가 최소화되도록 뉴턴-랩슨 반복 연산을 수행한다. 일반 의사역행렬($J^{\dagger}$)을 이용해 각 관절 각도를 보정한다.

$$
\Delta\theta = J^{\dagger} \mathbf{e}
$$
$$
\theta_{new} = \theta_{old} + \alpha \Delta\theta
$$

## 2. 특이점과 Damped Least Squares (DLS)

조작성이 매우 낮은 특이점 근처에서 관절 각속도가 비이상적으로 발산하는 폭주 문제를 감쇠 계수 $\lambda$를 삽입하여 극복하는 공식이다.

$$
J_{DLS} = J^T (J J^T + \lambda^2 I)^{-1}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class RobotArm3DOF:
    def __init__(self, L1=0.3, L2=0.3, L3=0.2):
        self.links = [L1, L2, L3]
        
    def fk(self, thetas):
        t1, t2, t3 = thetas
        L1, L2, L3 = self.links
        p0 = np.array([0.0, 0.0])
        p1 = p0 + np.array([L1 * np.cos(t1), L1 * np.sin(t1)])
        p2 = p1 + np.array([L2 * np.cos(t1 + t2), L2 * np.sin(t1 + t2)])
        p3 = p2 + np.array([L3 * np.cos(t1 + t2 + t3), L3 * np.sin(t1 + t2 + t3)])
        return p0, p1, p2, p3

    def get_analytical_jacobian(self, thetas):
        t1, t2, t3 = thetas
        L1, L2, L3 = self.links
        
        dx_dt1 = -L1*np.sin(t1) - L2*np.sin(t1+t2) - L3*np.sin(t1+t2+t3)
        dx_dt2 = -L2*np.sin(t1+t2) - L3*np.sin(t1+t2+t3)
        dx_dt3 = -L3*np.sin(t1+t2+t3)
        
        dy_dt1 = L1*np.cos(t1) + L2*np.cos(t1+t2) + L3*np.cos(t1+t2+t3)
        dy_dt2 = L2*np.cos(t1+t2) + L3*np.cos(t1+t2+t3)
        dy_dt3 = L3*np.cos(t1+t2+t3)
        
        return np.array([
            [dx_dt1, dx_dt2, dx_dt3],
            [dy_dt1, dy_dt2, dy_dt3]
        ])

    def ik_solver(self, target, initial_thetas, lam=0.0, max_iter=50, lr=0.5):
        """lam=0 이면 일반 의사역행렬, lam > 0 이면 DLS 수치 IK"""
        thetas = np.array(initial_thetas, dtype=float)
        error_history = []
        theta_history = [thetas.copy()]
        
        for _ in range(max_iter):
            _, _, _, p3 = self.fk(thetas)
            err = target - p3
            err_norm = np.linalg.norm(err)
            error_history.append(err_norm)
            
            if err_norm < 1e-5:
                break
                
            J = self.get_analytical_jacobian(thetas)
            
            # 의사역행렬 vs DLS 계산
            JJ_T = J @ J.T
            inv_part = np.linalg.inv(JJ_T + (lam**2) * np.eye(2))
            J_dls = J.T @ inv_part
            
            d_theta = J_dls @ err
            thetas += lr * d_theta
            theta_history.append(thetas.copy())
            
        return thetas, error_history, np.array(theta_history)

# 예제 1: 감쇠 계수(lambda) 값에 따른 역기구학 수렴 양상 비교
robot = RobotArm3DOF()
target = np.array([0.4, 0.4])
initial = np.radians([15, 15, 15])

_, err_none, _ = robot.ik_solver(target, initial, lam=0.0, lr=0.8)
_, err_dls_1, _ = robot.ik_solver(target, initial, lam=0.01, lr=0.8)
_, err_dls_2, _ = robot.ik_solver(target, initial, lam=0.1, lr=0.8)

plt.figure(figsize=(8, 4.5))
plt.plot(err_none, 'o-', label='No Damping (Pseudo-inverse, $\lambda=0.0$)', color='red')
plt.plot(err_dls_1, 's-', label='Light Damping ($\lambda=0.01$)', color='orange')
plt.plot(err_dls_2, '^-', label='Heavy Damping ($\lambda=0.1$)', color='blue')
plt.xlabel('Iterations')
plt.ylabel('Position Error Norm [m]')
plt.title('IK Convergence Comparison by Damping Coefficients ($\lambda$)', fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--')
plt.legend()
plt.show()

## 3. 특이점 경로 통과 시뮬레이션

로봇이 일자로 완전히 펴진 특이점 상태 부근의 직선 경로를 추적할 때, 일반 의사역행렬과 DLS 방식의 관절각 변화 속도 발산 억제 능력을 시뮬레이션으로 비교한다.

In [ ]:
# 예제 2: 특이점 경계선 근처(x=0.79 ~ 0.8) 구간 미소 직선 이동 시뮬레이션
path_x = np.linspace(0.70, 0.799, 30)
path_y = np.zeros_like(path_x) # y=0 고정 (x축 위에서 쭉 펴지는 특이점 방향)

# 초기 각도 (거의 펴진 포즈)
q = np.radians([1, 1, 1])

joint_velocities_pseudo = []
joint_velocities_dls = []

# 1) 일반 의사 역행렬로 추적 시 관절각 변화율 속도
q_pseudo = q.copy()
for i in range(len(path_x)):
    target_pt = np.array([path_x[i], path_y[i]])
    q_pseudo, _, hist = robot.ik_solver(target_pt, q_pseudo, lam=0.0, max_iter=8, lr=0.9)
    if len(hist) > 1:
        joint_velocities_pseudo.append(np.linalg.norm(hist[-1] - hist[-2]))
    else:
        joint_velocities_pseudo.append(0.0)

# 2) DLS(lambda = 0.05)로 추적 시 관절각 변화율 속도
q_dls = q.copy()
for i in range(len(path_x)):
    target_pt = np.array([path_x[i], path_y[i]])
    q_dls, _, hist = robot.ik_solver(target_pt, q_dls, lam=0.05, max_iter=8, lr=0.9)
    if len(hist) > 1:
        joint_velocities_dls.append(np.linalg.norm(hist[-1] - hist[-2]))
    else:
        joint_velocities_dls.append(0.0)

# 시각화
plt.figure(figsize=(9, 4.5))
plt.plot(path_x, joint_velocities_pseudo, 'ro-', label='Pseudo-inverse (Unstable near singularity)')
plt.plot(path_x, joint_velocities_dls, 'bo-', label='DLS ($\\lambda=0.05$, Stable)')
plt.yscale('log')
plt.xlabel('Target X position [m]')
plt.ylabel('Joint Step Change Norm (log scale)')
plt.title('Joint Velocity Spike near Singularity Point (X=0.8m)', fontsize=12, fontweight='bold')
plt.grid(True, which="both", linestyle=':')
plt.legend()
plt.show()